In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)
from sentence_transformers import SentenceTransformer



/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------
# 1. Connect to Qdrant
# -----------------------------
client = QdrantClient(
    host="localhost",
    port=6333
)

In [3]:
# -----------------------------
# 2. Load Embedding Model
# -----------------------------
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

# Embedding dimension
embedding_dim = model.get_sentence_embedding_dimension()

print(f"Embedding dimension: {embedding_dim}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3464.78it/s]
/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce MX330 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForce MX330 with CUDA capabili

Embedding dimension: 384


/tmp/ipykernel_19074/2406878881.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


In [4]:
# -----------------------------
# 3. Create Collection
# -----------------------------
collection_name = "documents"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.COSINE
    )
)

True

In [6]:
    # -----------------------------
# 4. Documents
# -----------------------------
documents = [
    "Vector databases store embeddings for semantic search.",
    "Qdrant is an open-source vector database.",
    "Embeddings convert text into numerical vectors.",
    "RAG combines retrieval with language models.",
    "HNSW is a popular ANN indexing algorithm."
]

# -----------------------------
# 5. Create Embeddings
# -----------------------------
embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    device = "cpu"
)


In [ ]:
# -----------------------------
# 6. Insert into Qdrant
# -----------------------------
points = []

for idx, (doc, vector) in enumerate(
    zip(documents, embeddings)
):
    points.append(
        PointStruct(
            id=idx,
            vector=vector.tolist(),
            payload={
                "text": doc
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Inserted {len(points)} documents.")

Inserted 5 documents.


In [10]:
from sentence_transformers import SentenceTransformer

query = "How do vector databases work?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    device = "cpu"
)

results = client.query_points(
    collection_name="documents",
    query=query_embedding.tolist(),
    limit=5
)

results

QueryResponse(points=[ScoredPoint(id=0, version=1, score=0.7822708, payload={'text': 'Vector databases store embeddings for semantic search.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=1, score=0.7483957, payload={'text': 'Qdrant is an open-source vector database.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2, version=1, score=0.6690911, payload={'text': 'Embeddings convert text into numerical vectors.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=1, score=0.6266521, payload={'text': 'RAG combines retrieval with language models.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4, version=1, score=0.6218395, payload={'text': 'HNSW is a popular ANN indexing algorithm.'}, vector=None, shard_key=None, order_value=None)])

In [13]:
for i in results:
    for j in i:
        for k in j:
            print(k)

p
o
i
n
t
s
id=0 version=1 score=0.7822708 payload={'text': 'Vector databases store embeddings for semantic search.'} vector=None shard_key=None order_value=None
id=1 version=1 score=0.7483957 payload={'text': 'Qdrant is an open-source vector database.'} vector=None shard_key=None order_value=None
id=2 version=1 score=0.6690911 payload={'text': 'Embeddings convert text into numerical vectors.'} vector=None shard_key=None order_value=None
id=3 version=1 score=0.6266521 payload={'text': 'RAG combines retrieval with language models.'} vector=None shard_key=None order_value=None
id=4 version=1 score=0.6218395 payload={'text': 'HNSW is a popular ANN indexing algorithm.'} vector=None shard_key=None order_value=None


In [16]:
results.points

[ScoredPoint(id=0, version=1, score=0.7822708, payload={'text': 'Vector databases store embeddings for semantic search.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=1, score=0.7483957, payload={'text': 'Qdrant is an open-source vector database.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=1, score=0.6690911, payload={'text': 'Embeddings convert text into numerical vectors.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=0.6266521, payload={'text': 'RAG combines retrieval with language models.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=1, score=0.6218395, payload={'text': 'HNSW is a popular ANN indexing algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

embedder = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10105.41it/s]


In [22]:
def retrieve(query):
    query_embedding = embedder.encode(query, device="cpu")

    results = client.query_points(
        collection_name="documents",
        query=query_embedding.tolist(),
        limit=5
    )

    docs = [r.payload["text"] for r in results.points]

    scores = reranker.predict(
        [(query, doc) for doc in docs],
        device="cpu"
    )

    ranked_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_docs[:3]

In [23]:
retrieve(query)

[('Embeddings convert text into numerical vectors.', np.float32(0.14330631)),
 ('RAG combines retrieval with language models.', np.float32(0.038898878)),
 ('Vector databases store embeddings for semantic search.',
  np.float32(0.019193886))]

In [ ]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)

client.create_collection(
    collection_name="documents",

    vectors_config={
        "dense": VectorParams(
            size=768,
            distance=Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

In [24]:
from fastembed import TextEmbedding
from fastembed import SparseTextEmbedding

dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 16.32it/s]


In [25]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)

client.create_collection(
    collection_name="hybrid_docs",

    vectors_config={
        "dense": VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

True

In [ ]:
documents = [
    "Qdrant uses HNSW indexing.",
    "PostgreSQL stores relational data.",
    "Vector databases enable semantic search.",
    "HNSW is an ANN algorithm."
]

In [27]:
dense_embeddings = list(
    dense_model.embed(documents)
)

sparse_embeddings = list(
    sparse_model.embed(documents)
)

In [28]:
from qdrant_client.models import PointStruct

points = []

for i, doc in enumerate(documents):

    points.append(
        PointStruct(
            id=i,

            vector={
                "dense":
                    dense_embeddings[i].tolist(),

                "sparse": {
                    "indices":
                        sparse_embeddings[i].indices.tolist(),

                    "values":
                        sparse_embeddings[i].values.tolist()
                }
            },

            payload={
                "text": doc
            }
        )
    )

client.upsert(
    collection_name="hybrid_docs",
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
query = "How does Qdrant use HNSW?"

In [30]:
query_dense = list(
    dense_model.embed([query])
)[0]

query_sparse = list(
    sparse_model.embed([query])
)[0]

In [31]:
dense_results = client.query_points(
    collection_name="hybrid_docs",

    using="dense",

    query=query_dense.tolist(),

    limit=2
)

In [33]:
from qdrant_client.models import SparseVector

In [34]:
sparse_results = client.query_points(
    collection_name="hybrid_docs",
    using="sparse",
    query=SparseVector(
        indices=query_sparse.indices.tolist(),
        values=query_sparse.values.tolist()
    ),
    limit=2
)

In [35]:
dense_results

QueryResponse(points=[ScoredPoint(id=0, version=1, score=0.87899196, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=1, score=0.73414433, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)])

In [36]:
dense_results.points

[ScoredPoint(id=0, version=1, score=0.87899196, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=0.73414433, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [37]:
sparse_results.points

[ScoredPoint(id=0, version=1, score=8.431368, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=2.817995, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [38]:
from qdrant_client.models import (
    Prefetch,
    FusionQuery,
    Fusion,
    SparseVector
)

results = client.query_points(
    collection_name="hybrid_docs",

    prefetch=[
        Prefetch(
            query=query_dense.tolist(),
            using="dense",
            limit=2
        ),

        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist()
            ),
            using="sparse",
            limit=2
        )
    ],

    query=FusionQuery(
        fusion=Fusion.RRF
    ),

    limit=10
)

for point in results.points:
    print(point.score)
    print(point.payload["text"])
    print("-" * 50)

1.0
Qdrant uses HNSW indexing.
--------------------------------------------------
0.6666667
HNSW is an ANN algorithm.
--------------------------------------------------
